# Dunnhumby 개인이력 N/V M2 300 epoch 5시드 보충

기존 seed 42·43·44 재사용하고 seed 45·46의 M1/M2만 새로 학습합니다. 신규상품 개발구간 DAY 684~690만 평가하며, 300 epoch 고정점의 5시드 결과를 합칩니다.

판정은 기존 역할 기준을 그대로 쓸니다: Recall@50과 가격·구매금액 가중 적중값@50의 평균차가 양수이고 각각 4/5 시드 이상 양수, NDCG@10 평균은 M1의 98% 이상입니다. 유의성·일반화·CLV 귀속을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SOURCE_COMMIT = '967e659c924e09ebac79f37180b2078752bc7950'
%cd /content
!rm -rf /content/clv-m2-history-budget-dev5
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-history-budget-dev5
%cd /content/clv-m2-history-budget-dev5
!git checkout -q $SOURCE_COMMIT
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT

In [ ]:
from pathlib import Path
import json
import torch
import clv_m2_history_budget_dev5_completion as completion

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
ROOT = Path('/content/drive/MyDrive/논문/data')
OUT_DIR = ROOT / 'results_v3_dunnhumby_clv_m2_history_budget_dev5_completion_v1'
cfg = completion.configure(out_dir=str(OUT_DIR))
assert cfg.seeds == (45, 46)
assert cfg.epochs == 300 and cfg.conditions == ('baseline',)
print(json.dumps(completion.preflight_summary(cfg), ensure_ascii=False, indent=2))
paths = completion.run(cfg, root=ROOT)
print(json.dumps(paths, ensure_ascii=False, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(paths['summary']))
display(pd.read_csv(paths['paired']))